### 3.4.3. Quadratically Constrained Quadratic Programming (QCQP)

$$
\begin{aligned}
\min_{\mathbf{x}\in\mathbb{R}^n}\quad & \tfrac12\mathbf{x}^\top P_0 \mathbf{x} + \mathbf{q}_0^\top \mathbf{x} + r_0 \\
\text{subject to}\quad & \tfrac12\mathbf{x}^\top P_i \mathbf{x} + \mathbf{q}_i^\top \mathbf{x} + r_i \le 0,\quad i=1,\dots,m,
\end{aligned}
$$
with $P_0,\dots,P_m \succeq 0$ for a convex QCQP.

**Explanation:**

A quadratically constrained quadratic program (QCQP) extends [quadratic programming](./02_quadratic_programming.ipynb) by allowing the constraints themselves to be convex quadratics, so the feasible set is an intersection of ellipsoids rather than a polyhedron. When every $P_i$ is positive semidefinite the problem is convex and every local solution is global. A QCQP with $P_0=0$ (linear objective) and a single ellipsoidal constraint is the simplest nontrivial case, and every convex QCQP can be recast as a [second-order cone program](./04_second_order_cone_programming.ipynb).

**Intuition:**

The quadratic constraint bends the feasible boundary into an ellipsoid; the optimizer slides along that curved surface until the objective can decrease no further.

![Quadratic objective capped by an ellipsoidal constraint](../../Figures/030306_qcqp_polyhedron_ellipsoid_cap.png)

**Numerical Example:**

Minimize a linear objective over the unit disk:
$$
\min_{x_1,x_2}\; -x_1 - x_2
\quad\text{subject to}\quad x_1^2 + x_2^2 \le 1.
$$

With multiplier $\lambda \ge 0$ the Lagrangian is $L = -x_1 - x_2 + \lambda(x_1^2+x_2^2-1)$. Stationarity gives
$$
-1 + 2\lambda x_1 = 0,\quad -1 + 2\lambda x_2 = 0 \;\Rightarrow\; x_1 = x_2 = \tfrac{1}{2\lambda}.
$$
The constraint is active, so $x_1^2+x_2^2 = 1$ forces $\lambda = \tfrac{1}{\sqrt2}$ and
$$
\mathbf{x}^\star = \left(\tfrac{1}{\sqrt2}, \tfrac{1}{\sqrt2}\right),
\qquad f^\star = -\sqrt2 \approx -1.414.
$$

In [1]:
import sympy as sp

x_1, x_2, lam = sp.symbols("x_1 x_2 lambda", positive=True)
objective = -x_1 - x_2
constraint = x_1**2 + x_2**2 - 1
lagrangian = objective + lam*constraint

stationarity = [sp.diff(lagrangian, variable) for variable in (x_1, x_2)]
active_constraint = sp.Eq(constraint, 0)
solution = sp.solve(stationarity + [active_constraint], [x_1, x_2, lam], dict=True)[0]
minimizer = (solution[x_1], solution[x_2])
minimum_value = sp.simplify(objective.subs(solution))

print("minimizer =", minimizer)
print("multiplier lambda =", solution[lam])
print("minimum value =", minimum_value)

minimizer = (sqrt(2)/2, sqrt(2)/2)
multiplier lambda = sqrt(2)/2
minimum value = -sqrt(2)


**Equivalent casadi implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x", 2)
objective = -decision_variable[0] - decision_variable[1]
constraint = decision_variable[0]**2 + decision_variable[1]**2 - 1

solver = ca.nlpsol("solver", "ipopt", {"x": decision_variable, "f": objective, "g": constraint},
                   {"ipopt.print_level": 0, "print_time": 0, "ipopt.sb": "yes"})
solution = solver(x0=[0.1, 0.1], lbg=-ca.inf, ubg=0)

print("minimizer =", solution["x"])
print("minimum value =", float(solution["f"]))

minimizer = [0.707107, 0.707107]
minimum value = -1.4142135669370859


**References:**

[📘 Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.](https://web.stanford.edu/~boyd/cvxbook/)

---

[⬅️ Previous: Quadratic Programming (QP)](./02_quadratic_programming.ipynb) | [Next: Second-Order Cone Programming (SOCP) ➡️](./04_second_order_cone_programming.ipynb)